## Imports & Configurations

In [1]:
from astropy.table import Table
import numpy as np
import pandas as pd
import os
import warnings

## File Path

In [2]:
warnings.filterwarnings('ignore')
data_path = os.path.join(os.getcwd(), '..', 'Data')

## Load DR16Q_v4

In [3]:
qso_filename = os.path.join(data_path, 'DR16Q_v4.fits')
qso_data = Table.read(qso_filename, format='fits', hdu=1)

u_qso=qso_data['PSFMAG'][:,0]
g_qso=qso_data['PSFMAG'][:,1]
r_qso=qso_data['PSFMAG'][:,2]
i_qso=qso_data['PSFMAG'][:,3]
z_qso=qso_data['PSFMAG'][:,4]

u_qso_err = qso_data['PSFMAGERR'][:, 0]
g_qso_err = qso_data['PSFMAGERR'][:, 1]
r_qso_err = qso_data['PSFMAGERR'][:, 2]
i_qso_err = qso_data['PSFMAGERR'][:, 3]
z_qso_err = qso_data['PSFMAGERR'][:, 4]

u_qso_extinction = qso_data['EXTINCTION'][:, 0]
g_qso_extinction = qso_data['EXTINCTION'][:, 1]
r_qso_extinction = qso_data['EXTINCTION'][:, 2]
i_qso_extinction = qso_data['EXTINCTION'][:, 3]
z_qso_extinction = qso_data['EXTINCTION'][:, 4]

u_clean_qso = u_qso - u_qso_extinction
g_clean_qso = g_qso - g_qso_extinction
r_clean_qso = r_qso - r_qso_extinction
i_clean_qso = i_qso - i_qso_extinction
z_clean_qso = z_qso - z_qso_extinction

## Quasar Quality Cuts

In [4]:
n_total_qso = len(qso_data)

real_quasars = ((qso_data['IS_QSO_FINAL'] == 1) & (qso_data['THING_ID'] != -1) & (qso_data['Z'] != -999) & (qso_data['Z_PCA'] >= 0))
n_after_real = np.count_nonzero(real_quasars)

good_spectrum = real_quasars & (qso_data['ZWARNING'] == 0) & (qso_data['ZWARN_PCA'] == 0)
n_after_spectrum = np.count_nonzero(good_spectrum)

suspect = (qso_data['SOURCE_Z'] == 'PIPE') & (qso_data['Z'] > 5)
not_suspect = good_spectrum & (~suspect)
n_after_suspect = np.count_nonzero(not_suspect)

mask_photometry = ((u_qso_err < 0.1) & (g_qso_err < 0.1) & (r_qso_err < 0.1) &
                    (i_qso_err < 0.1) & (z_qso_err < 0.1))
after_photometry = not_suspect & mask_photometry
n_after_photometry = np.count_nonzero(after_photometry)

mask_valid_values = ((u_clean_qso > -100) & (u_clean_qso < 100) &
                     (g_clean_qso > -100) & (g_clean_qso < 100) &
                     (r_clean_qso > -100) & (r_clean_qso < 100) &
                     (i_clean_qso > -100) & (i_clean_qso < 100) &
                     (z_clean_qso > -100) & (z_clean_qso < 100))
master_mask = after_photometry & mask_valid_values
n_after_valid = np.count_nonzero(master_mask)

r_i_clean_qso = r_clean_qso[master_mask] - i_clean_qso[master_mask]
g_r_clean_qso = g_clean_qso[master_mask] - r_clean_qso[master_mask]
u_g_clean_qso = u_clean_qso[master_mask] - g_clean_qso[master_mask]
i_z_clean_qso = i_clean_qso[master_mask] - z_clean_qso[master_mask]

## Selected Tiles for Star Catalog

In [5]:
ra = np.asarray(qso_data[master_mask]['RA'])
dec = np.asarray(qso_data[master_mask]['DEC'])

ra_edges = np.arange(0, 361, 10)
dec_edges = np.arange(-90, 91, 10)

ra_tile = np.digitize(ra, ra_edges) - 1
dec_tile = np.digitize(dec, dec_edges) - 1

tiles = pd.DataFrame({'ra_tile': ra_tile, 'dec_tile': dec_tile})

tile_counts = (
    tiles.groupby(['ra_tile', 'dec_tile'])
    .size()
    .reset_index(name='N_quasars')
    .sort_values('N_quasars', ascending=False)
)

tile_counts['ra_min'] = ra_edges[tile_counts['ra_tile']]
tile_counts['ra_max'] = ra_edges[tile_counts['ra_tile'] + 1]
tile_counts['dec_min'] = dec_edges[tile_counts['dec_tile']]
tile_counts['dec_max'] = dec_edges[tile_counts['dec_tile'] + 1]

selected_tiles = tile_counts.head(20).copy()

selected_tiles[['ra_min','ra_max','dec_min','dec_max','N_quasars']]

,ra_min,ra_max,dec_min,dec_max,N_quasars
13,20,30,0,10,2653
7,10,20,0,10,2549
2,0,10,0,10,2542
43,130,140,20,30,2391
9,10,20,20,30,2339
60,150,160,30,40,2214
162,350,360,0,10,2182
52,140,150,30,40,2182
36,120,130,30,40,2090
158,340,350,20,30,2064


In [6]:
def tile_area_deg2(ra_min, ra_max, dec_min, dec_max):
    dec_min_rad = np.radians(dec_min)
    dec_max_rad = np.radians(dec_max)
    return (ra_max - ra_min) * (np.sin(dec_max_rad) - np.sin(dec_min_rad)) * (180 / np.pi)

selected_tiles['area_deg2'] = selected_tiles.apply(
    lambda row: tile_area_deg2(row['ra_min'], row['ra_max'], row['dec_min'], row['dec_max']),
    axis=1
)

total_area = selected_tiles['area_deg2'].sum()
print(f'Total sky coverage: {total_area:.1f} deg²')
print(selected_tiles[['ra_min','ra_max','dec_min','dec_max','area_deg2']])

Total sky coverage: 1740.5 deg²
     ra_min  ra_max  dec_min  dec_max  area_deg2
13       20      30        0       10  99.493077
7        10      20        0       10  99.493077
2         0      10        0       10  99.493077
43      130     140       20       30  90.515790
9        10      20       20       30  90.515790
60      150     160       30       40  81.811274
162     350     360        0       10  99.493077
52      140     150       30       40  81.811274
36      120     130       30       40  81.811274
158     340     350       20       30  90.515790
59      150     160       20       30  90.515790
131     240     250       30       40  81.811274
137     250     260       30       40  81.811274
109     210     220       40       50  70.620964
100     200     210       30       40  81.811274
14       20      30       10       20  96.470030
92      190     200       30       40  81.811274
132     240     250       40       50  70.620964
17       30      40      -10        0

The 20 selected tiles above were used to construct the spatial WHERE clause
in `SQL/star_catalog_query.sql`, which was submitted to SDSS CasJobs to
download the corresponding star catalog (see `Data/README.md` for download
instructions).

## Load Star Catalog (From CASJobs)

In [7]:
stars_filename = os.path.join(data_path, 'Star_Catalog.fit')
stars_data = Table.read(stars_filename, format='fits', hdu=1)

u_star = stars_data['psfMag_u']
g_star = stars_data['psfMag_g']
r_star = stars_data['psfMag_r']
i_star = stars_data['psfMag_i']
z_star = stars_data['psfMag_z']

u_star_err = stars_data['psfMagErr_u']
g_star_err = stars_data['psfMagErr_g']
r_star_err = stars_data['psfMagErr_r']
i_star_err = stars_data['psfMagErr_i']
z_star_err = stars_data['psfMagErr_z']

u_star_extinction = stars_data['extinction_u']
g_star_extinction = stars_data['extinction_g']
r_star_extinction = stars_data['extinction_r']
i_star_extinction = stars_data['extinction_i']
z_star_extinction = stars_data['extinction_z']

u_clean_star = u_star - u_star_extinction
g_clean_star = g_star - g_star_extinction
r_clean_star = r_star - r_star_extinction
i_clean_star = i_star - i_star_extinction
z_clean_star = z_star - z_star_extinction

r_i_clean_star = r_clean_star - i_clean_star
g_r_clean_star = g_clean_star - r_clean_star
u_g_clean_star = u_clean_star - g_clean_star
i_z_clean_star = i_clean_star - z_clean_star

## Additional Star Quality Cuts

In [8]:
mask_valid_colors_star = ((r_i_clean_star > -2) & (r_i_clean_star < 5) &
                           (g_r_clean_star > -2) & (g_r_clean_star < 5) &
                           (u_g_clean_star > -2) & (u_g_clean_star < 5) &
                           (i_z_clean_star > -2) & (i_z_clean_star < 5))

n_before_color_cut = len(r_i_clean_star)

r_i_clean_star = r_i_clean_star[mask_valid_colors_star]
g_r_clean_star = g_r_clean_star[mask_valid_colors_star]
u_g_clean_star = u_g_clean_star[mask_valid_colors_star]
i_z_clean_star = i_z_clean_star[mask_valid_colors_star]

n_after_color_cut = len(r_i_clean_star)
print('Stars before color cut:', n_before_color_cut)
print('Stars after color cut:', n_after_color_cut)
print('Rejected:', n_before_color_cut - n_after_color_cut)

Stars before color cut: 1906080
Stars after color cut: 1904209
Rejected: 1871


## Export Quasar Quality Cuts

In [9]:
qso_cuts = [
    {'step': 0, 'description': 'total_quasars_in_catalog', 'rejected': None, 'remaining': n_total_qso},
    {'step': 1, 'description': 'valid_quasar_classification_and_redshift', 'rejected': n_total_qso - n_after_real, 'remaining': n_after_real},
    {'step': 2, 'description': 'good_spectrum_zwarning', 'rejected': n_after_real - n_after_spectrum, 'remaining': n_after_spectrum},
    {'step': 3, 'description': 'exclude_suspect_pipeline_redshifts', 'rejected': n_after_spectrum - n_after_suspect, 'remaining': n_after_suspect},
    {'step': 4, 'description': 'photometric_uncertainty_below_0.1', 'rejected': n_after_suspect - n_after_photometry, 'remaining': n_after_photometry},
    {'step': 5, 'description': 'valid_psf_magnitudes', 'rejected': n_after_photometry - n_after_valid, 'remaining': n_after_valid},
]

qso_cuts_df = pd.DataFrame(qso_cuts)
output_path = os.path.join(os.getcwd(), '..', 'Outputs', 'Quasar_Selection_Cuts.csv')
qso_cuts_df.to_csv(output_path, index=False)
qso_cuts_df

,step,description,rejected,remaining
0,0,total_quasars_in_catalog,NaN,750414
1,1,valid_quasar_classification_and_redshift,1759.0,748655
2,2,good_spectrum_zwarning,111268.0,637387
3,3,exclude_suspect_pipeline_redshifts,158.0,637229
4,4,photometric_uncertainty_below_0.1,474349.0,162880
5,5,valid_psf_magnitudes,3.0,162877


## Export Additional Star Quality Cuts

In [10]:
star_cuts = [
    {'step': 0, 'description': 'total_stars_in_casjobs_catalog', 'rejected': None, 'remaining': n_before_color_cut},
    {'step': 1, 'description': 'valid_stellar_color_range', 'rejected': n_before_color_cut - n_after_color_cut, 'remaining': n_after_color_cut},
]

star_cuts_df = pd.DataFrame(star_cuts)
star_output_path = os.path.join(os.getcwd(), '..', 'Outputs', 'Star_Selection_Cuts_Python.csv')
star_cuts_df.to_csv(star_output_path, index=False)
star_cuts_df

,step,description,rejected,remaining
0,0,total_stars_in_casjobs_catalog,NaN,1906080
1,1,valid_stellar_color_range,1871.0,1904209


## Export Arrays

In [11]:
np.savez(os.path.join(os.getcwd(), '..', 'Outputs', 'Quasar_Colors.npz'),
         r_i_clean_qso=r_i_clean_qso,
         g_r_clean_qso=g_r_clean_qso,
         u_g_clean_qso=u_g_clean_qso,
         i_z_clean_qso=i_z_clean_qso,
         Z=qso_data['Z_PCA'][master_mask])

np.savez(os.path.join(os.getcwd(), '..', 'Outputs', 'Star_Colors.npz'),
         r_i_clean_star=r_i_clean_star,
         g_r_clean_star=g_r_clean_star,
         u_g_clean_star=u_g_clean_star,
         i_z_clean_star=i_z_clean_star)